In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
# Permite importar tu paquete de src/ sin instalarlo
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))


# 04 · Variables — cada idea es una hipótesis que se mide

> **Esqueleto.** Guía completa: `curso/kit-estudiante/guia-modelado-y-optimizacion.md`, sección 1.

Crear variables es fácil; saber si **sirven** es lo difícil. Cada idea se mide con **el mismo modelo, los
mismos pliegues y la misma métrica**; solo cambia el conjunto de variables. Una idea entra solo si la
mejora supera el **ruido entre pliegues**.

Qué produce este notebook: una **tabla de ideas medidas** con su decisión, y el código reutilizable en
`src/proyecto/features/`. Este notebook solo **orquesta**.

## 0. Cargar los datos de entrenamiento

In [2]:
import pandas as pd

# === Configuración: completa estas dos líneas (las mismas de tu notebook 03) ===
ID_COL = "cole_cod_dane_sede"   # sede del colegio: la entidad que se repite (decisión 10 del EDA)
TARGET = "punt_global"   # puntaje global de Saber 11

train_path = Path.cwd().resolve().parent / "data" / "processed" / "train.joblib"
if not train_path.exists():
    train = None
    print("Todavía no hay datos de entrenamiento: completa y ejecuta antes el notebook 03_preprocesamiento.")
else:
    import joblib
    train = joblib.load(train_path)
    print(f"entrenamiento: {train.shape[0]:,} filas x {train.shape[1]} columnas (la prueba NO se abre hoy)")

listo = train is not None and TARGET is not None
if train is not None and TARGET is None:
    print("Define TARGET (y, si hay entidades repetidas, ID_COL) arriba para continuar.")

entrenamiento: 57,286 filas x 36 columnas (la prueba NO se abre hoy)


## 1. Mis ideas, antes de medirlas

Las ideas vienen de `especificaciones/04-features.md` (sección 3) y de la decisión del grupo sobre municipios. Todas usan
solo columnas de la misma fila que se conocen **antes** del examen.

| # | Idea (variable y cómo se calcula) | Por qué podría ayudar | ¿Se conoce al predecir? |
|---|---|---|---|
| 1 | `n_bienes_hogar`: número de «Si» entre automóvil, computador, internet y lavadora (0 a 4) | Resume la situación económica del hogar | ☒ sí ☐ no |
| 2 | `educacion_max_padres`: nivel más alto entre madre y padre (escala 0–9 aprobada por el grupo) | Importa el nivel educativo del hogar, no el de cada padre por separado | ☒ sí ☐ no |
| 3 | `personas_por_cuarto`: punto medio de personas ÷ cuartos | Mide el hacinamiento | ☒ sí ☐ no |
| 4 | `vive_en_mcpio_colegio`: 1 si el municipio de residencia es el del colegio | Desplazamiento para estudiar | ☒ sí ☐ no |
| 5 | Las 4 juntas | Pueden aportar juntas aunque por separado no se note | ☒ sí ☐ no |
| 6 | Umbral de poco frecuentes `min_frequency` = 0,5 % y 0,1 % (en vez de 1 %) | Más municipios con columna propia (decisión del grupo: probar otros umbrales). Ojo: el umbral aplica a **todas** las categóricas, no solo a los municipios | ☒ sí ☐ no |

**Regla de decisión (escrita ANTES de medir):** conservo una idea solo si **mejora / ruido > 1** (la mejora del MAE
supera el ruido entre pliegues, como pide `reports/04_features.md`) y se conoce al predecir. Si ninguna la cumple, las
variables nuevas quedan **apagadas** y el umbral queda en 1 % (`especificaciones/04-features.md`).

## 2. Validación común

Los pliegues se calculan **una vez** y se reutilizan en toda comparación. Si hay entidades repetidas, los
pliegues son **por entidad**.

In [3]:
if not listo:
    pliegues = None
    print("Sin datos o sin TARGET: esta celda crea los pliegues cuando estén.")
else:
    from sklearn.model_selection import GroupKFold, KFold, StratifiedGroupKFold, StratifiedKFold

    X, y = train.drop(columns=[TARGET]), train[TARGET]
    es_regresion = pd.api.types.is_numeric_dtype(y) and y.nunique() > 20   # ajusta si tu objetivo es otro
    if ID_COL is not None:
        cv = GroupKFold(n_splits=5) if es_regresion else StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
        pliegues = list(cv.split(X, y, groups=X[ID_COL]))
        for i_ent, i_val in pliegues:   # comprobación: ninguna entidad en los dos lados de un pliegue
            assert set(X.iloc[i_ent][ID_COL]).isdisjoint(set(X.iloc[i_val][ID_COL])), "hay entidades en los dos lados"
    else:
        cv = KFold(n_splits=5, shuffle=True, random_state=42) if es_regresion else StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        pliegues = list(cv.split(X, y))
    print(f"{len(pliegues)} pliegues, los MISMOS para todo lo que compares hoy.")

5 pliegues, los MISMOS para todo lo que compares hoy.


## 3. Pipeline base y métrica

Importa tu función de `src/proyecto/models/train.py` y define la métrica. La **métrica principal** debe
reflejar la decisión del negocio (tu ficha de caso); añade 1–2 **de control**.

In [4]:
try:
    from proyecto.models.train import build_pipeline, scoring as metricas_cv
except ModuleNotFoundError:
    build_pipeline = None
    print("No se pudo importar build_pipeline de src/proyecto/models/train.py: ejecuta el notebook desde notebooks/.")

scoring = metricas_cv()   # MAE (principal), RMSE y R²; scikit-learn da MAE y RMSE en negativo
METRICA = "mae"   # se lee con .abs() en la sección 5: queda en puntos, positivo

## 4. El experimento

Un modelo con **parámetros fijos** y una variante por idea. Define en `variantes` qué columnas usa cada una
(la referencia primero).

In [5]:
from sklearn.ensemble import HistGradientBoostingRegressor

from proyecto.features.engineering import NUMERIC_ENGINEERED, add_features

# Modelo fijo: el boosting del torneo con sus parámetros de partida (equivalentes a LightGBM del ejemplo).
modelo_fijo = HistGradientBoostingRegressor(
    max_iter=300, learning_rate=0.05, max_leaf_nodes=31, early_stopping=False, random_state=42
)

# Las variables nuevas se calculan una vez (solo usan la misma fila, sin aprender de los datos) y cada variante
# elige cuáles usa con `extra_numeric`. La referencia las ignora.
X_ideas = add_features(X) if listo else None

variantes = {
    "referencia": {},
    "+ n_bienes_hogar": {"extra_numeric": ["n_bienes_hogar"]},
    "+ educacion_max_padres": {"extra_numeric": ["educacion_max_padres"]},
    "+ personas_por_cuarto": {"extra_numeric": ["personas_por_cuarto"]},
    "+ vive_en_mcpio_colegio": {"extra_numeric": ["vive_en_mcpio_colegio"]},
    "+ las 4 juntas": {"extra_numeric": NUMERIC_ENGINEERED},
    "min_frequency 0,5 %": {"min_frequency": 0.005},
    "min_frequency 0,1 %": {"min_frequency": 0.001},
}
resultados = {}   # nombre de variante -> DataFrame con una fila por pliegue

faltan = [nombre for nombre, valor in {"pliegues": pliegues, "build_pipeline": build_pipeline, "scoring": scoring,
                                          "modelo_fijo": modelo_fijo, "variantes": variantes}.items()
          if valor is None or (hasattr(valor, "__len__") and len(valor) == 0)]
if faltan:
    print("Pendiente: completa", ", ".join(faltan), "para correr el experimento.")
else:
    from sklearn.base import clone
    from sklearn.model_selection import cross_validate

    for nombre, opciones in variantes.items():
        pipeline = build_pipeline(clone(modelo_fijo), **opciones)
        res = cross_validate(pipeline, X_ideas, y, cv=pliegues, scoring=scoring, n_jobs=1)
        resultados[nombre] = pd.DataFrame({k.replace("test_", ""): v for k, v in res.items() if k.startswith("test_")})
        print(f"{nombre:30s} listo")

referencia                     listo


+ n_bienes_hogar               listo


+ educacion_max_padres         listo


+ personas_por_cuarto          listo


+ vive_en_mcpio_colegio        listo


+ las 4 juntas                 listo


min_frequency 0,5 %            listo


min_frequency 0,1 %            listo


## 5. Mejora sobre ruido

In [6]:
import numpy as np


def mejora_sobre_ruido(referencia, variante):
    """Valores por pliegue de una métrica donde MENOR es mejor. Positivo: la variante mejora."""
    mejora = np.mean(referencia) - np.mean(variante)
    ruido = np.std(variante, ddof=1)
    return mejora, ruido, (mejora / ruido if ruido > 0 else np.inf)


if not resultados or METRICA is None:
    print("Sin resultados todavía: esta celda construirá tu tabla de mejora / ruido.")
else:
    nombres = list(resultados)
    ref = resultados[nombres[0]][METRICA].abs()   # MAE en positivo (scikit-learn lo da negativo)
    filas = []
    for nombre in nombres:
        valores = resultados[nombre][METRICA].abs()
        mejora, ruido, razon = mejora_sobre_ruido(ref, valores)
        filas.append({"variante": nombre, "métrica": valores.mean(), "± ruido": ruido, "mejora": mejora, "mejora / ruido": razon})
    tabla = pd.DataFrame(filas).set_index("variante")
    display(tabla.round(4))

,métrica,± ruido,mejora,mejora / ruido
variante,,,,
referencia,33.5630,0.2437,0.0000,0.0000
+ n_bienes_hogar,33.5658,0.2564,-0.0028,-0.0109
+ educacion_max_padres,33.6137,0.2786,-0.0507,-0.1822
+ personas_por_cuarto,33.5477,0.2688,0.0153,0.0568
+ vive_en_mcpio_colegio,33.5615,0.2294,0.0014,0.0063
+ las 4 juntas,33.5791,0.2770,-0.0161,-0.0582
"min_frequency 0,5 %",33.5033,0.2298,0.0597,0.2599
"min_frequency 0,1 %",33.3895,0.2257,0.1735,0.7686


## 6. Decisiones

Una fila por idea, **con su cifra** (mejora / ruido). Cópiala a `reports/04_features.md`.

| # | Idea | MAE (± ruido) | Mejora / ruido | Decisión | Motivo |
|---|---|---|---|---|---|
| 0 | Referencia (variables base, umbral 1 %) | 33,563 (± 0,244) | 0 | — | — |
| 1 | `n_bienes_hogar` | 33,566 (± 0,256) | −0,01 | ☐ conservar ☒ descartar | No mejora: el MAE sube 0,003 puntos |
| 2 | `educacion_max_padres` | 33,614 (± 0,279) | −0,18 | ☐ conservar ☒ descartar | Empeora 0,05 puntos |
| 3 | `personas_por_cuarto` | 33,548 (± 0,269) | 0,06 | ☐ conservar ☒ descartar | Mejora 0,015 puntos, muy por debajo del ruido |
| 4 | `vive_en_mcpio_colegio` | 33,562 (± 0,229) | 0,01 | ☐ conservar ☒ descartar | Mejora 0,001 puntos: dentro del ruido |
| 5 | Las 4 juntas | 33,579 (± 0,277) | −0,06 | ☐ conservar ☒ descartar | Juntas tampoco mejoran |
| 6a | `min_frequency` 0,5 % | 33,503 (± 0,230) | 0,26 | ☐ conservar ☒ descartar | Mejora 0,06 puntos, dentro del ruido |
| 6b | `min_frequency` 0,1 % | 33,390 (± 0,226) | 0,77 | ☐ conservar ☒ descartar | Mejora 0,17 puntos, la mayor, pero no supera el ruido (0,77 < 1). Candidato a probarse como hiperparámetro en la etapa 6 |

Las cuatro variables nuevas no aportan: la información que resumen (bienes, educación, hogar, municipio) ya está en las
columnas base. Quedan **apagadas** (`use_engineered=False`) y el umbral queda en 1 %, como en el ejemplo del curso.

> **Una idea descartada con su cifra es un resultado, no un fracaso.** Anótala igual: documenta lo que ya
> probaste.

## Entregable de esta etapa

- [x] Cada idea tiene su fila con **mejora / ruido** y una decisión escrita.
- [x] Todas las variantes se midieron con **los mismos pliegues** (por entidad si hay entidades repetidas).
- [x] Ninguna variable usa información que no se conoce **al predecir**.
- [x] El código reutilizable está en `src/proyecto/features/` y el notebook solo lo llama.
- [x] `reports/04_features.md` completado.